In [ ]:
import sys
sys.path.insert(0, '.')
import figio as fx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# plot3: ROC AUC against parameter count. The vertical bar is the SD over the
# three seeds, which is what decides whether two points on this curve are
# actually distinguishable.
df = fx.rep('1_whole', plotted_only=False)
df = df[df['params'].notna() & df['roc_auc'].notna() & (df['params'] > 39450)]

plt.figure(figsize=(6, 5))
for fam, g in df.groupby('fam'):
    g = g.sort_values('params')
    plt.errorbar(g['params'], g['roc_auc'], yerr=g['roc_auc_sd'].fillna(0),
                 marker='o', capsize=3, label=fam, lw=1.4, elinewidth=0.9)
    for _, row in g.iterrows():
        if _ <4 :
            plt.text(row['params'] * 1.08, row['roc_auc'], row['full_name'], ha='left', va='center', fontsize=8)
        elif _ == 4:
            plt.text(row['params'] /1.08, row['roc_auc'], row['full_name'], ha='right', va='center', fontsize=8)
        elif _ == 5:
            plt.text(row['params'], row['roc_auc']+0.0015, row['full_name'], ha='center', va='bottom', fontsize=8)
        elif _ == 6:
            plt.text(row['params'], row['roc_auc']-0.0015, row['full_name'], ha='center', va='top', fontsize=8)
        elif _ == 7:
            plt.text(row['params'], row['roc_auc']+0.0015, row['full_name'], ha='left', va='bottom', fontsize=8)
        elif _ == 8:
            plt.text(row['params']*1.08, row['roc_auc']-0.0003, row['full_name'], ha='left', va='top', fontsize=8)
        elif _ == 9:  # label only the second and third rows
            plt.text(row['params']/1.08, row['roc_auc']+0.0003, row['full_name'], ha='right', va='bottom', fontsize=8)
        else:
            plt.text(row['params'], row['roc_auc'], row['full_name'], ha='center', va='top', fontsize=8)

plt.xscale('log')
plt.xlabel('Parameters')
plt.ylabel('ROC AUC')
plt.grid(True, which='both', linestyle='-', linewidth=0.1)

desired = ['Baseline', 'AF3-like', 'ESMC', 'ESM3']
h, l = plt.gca().get_legend_handles_labels()
order = [l.index(f) for f in desired if f in l]
plt.legend([h[i] for i in order], [l[i] for i in order],
           loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.08), frameon=False)
plt.tight_layout()
plt.savefig('fig5.pdf', bbox_inches='tight')
plt.savefig('fig5.svg', bbox_inches='tight')
plt.show()

In [ ]:
# The same points as a table. Where two models' intervals overlap, the scaling
# curve is not separating them.
t = df[['full_name', 'fam', 'params', 'roc_auc', 'roc_auc_sd', 'n_seeds']].copy()
t['reported'] = t.apply(lambda r: f"{r.roc_auc:.4f} +- {r.roc_auc_sd:.4f}", axis=1)
t.sort_values('roc_auc', ascending=False)[['full_name', 'fam', 'params', 'reported', 'n_seeds']]